### Imports + load + sample

In [3]:
import sys
sys.path.insert(0, "..")

In [4]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split

from src.fe_v2 import make_features
from src.config import RANDOM_SEED, TEST_SIZE, TOP_K
from src.metrics import mapk, hit_rate_at_k
from src.model_utils import topk_from_proba

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DATA_PATH = "../data/processed/df_model.parquet"
df = pd.read_parquet(DATA_PATH)

# Start smaller first (recommended)
df = df.sample(n=100_000, random_state=RANDOM_SEED).reset_index(drop=True)

X, y = make_features(df)

print(X.shape, y.shape)
print(y.nunique())


(100000, 166) (100000,)
100


### Split train/val/test (stratified)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)

# make a validation split from train
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=RANDOM_SEED, stratify=y_train
)

print(X_train.shape, X_val.shape, X_test.shape)


(60000, 166) (15000, 166) (25000, 166)


### Define feature groups

Neural networks are sensitive to how inputs are represented.

Categorical ID-like features (e.g., srch_destination_id) should NOT be scaled; we represent them using embeddings.

Numeric/continuous features (counts, booleans, and destination latent features d1..d150) are normalized (similar to standard scaling).

This cell creates:

- cat_cols: categorical columns → lookup + embedding

- num_cols: numeric columns → normalization

- dest_cols: d1..d150 detected with regex and treated as numeric

In [6]:
import re

# categorical features (IDs and engineered buckets)
cat_cols = [
    "site_name",
    "posa_continent",
    "user_location_country",
    "user_location_region",
    "srch_destination_id",
    "srch_destination_type_id",
    "channel",
    "stay_type",
    "distance_bucket",
]

# numeric features
num_cols = [
    "srch_adults_cnt",
    "srch_children_cnt",
    "srch_rm_cnt",
    "checkin_month",
    "length_of_stay",
    "is_mobile",
    "is_package",
    "distance_missing",
]

# destination latent continuous features d1..d150
dest_cols = [c for c in X.columns if re.fullmatch(r"d\d+", c)]
num_cols = num_cols + sorted(dest_cols)

# quick sanity:
missing = set(X.columns) - set(cat_cols) - set(num_cols)
print("Unassigned columns:", missing)


Unassigned columns: set()


### Build TF preprocessing: lookup+embedding + normalization

Build preprocessing layers inside the model

We create a Keras functional model with separate inputs for each column:

For each categorical column:

- StringLookup or IntegerLookup converts raw values into integer indices

- Embedding maps each category index into a dense learned vector

We flatten the embedding to a 1D vector

For numeric columns:

- We concatenate them into one numeric tensor

- Apply Normalization() fitted (“adapted”) only on training data

Finally, we concatenate all embedded categorical vectors + normalized numeric features into one combined feature representation for the DNN.

In [7]:
from tensorflow.keras import layers

# --- Inputs ---
inputs = {}
for c in cat_cols:
    # strings vs ints: stay_type/distance_bucket are objects, ids are ints
    dtype = tf.string if X[c].dtype == "object" else tf.int64
    inputs[c] = tf.keras.Input(shape=(1,), name=c, dtype=dtype)

for c in num_cols:
    inputs[c] = tf.keras.Input(shape=(1,), name=c, dtype=tf.float32)

# --- Categorical: Lookup -> Embedding -> Flatten ---
encoded_cats = []
for c in cat_cols:
    if X[c].dtype == "object":
        lookup = layers.StringLookup(output_mode="int", name=f"{c}_lookup")
        lookup.adapt(X_train[c].astype(str).values)  # adapt on train only
    else:
        lookup = layers.IntegerLookup(output_mode="int", name=f"{c}_lookup")
        lookup.adapt(X_train[c].values)

    vocab_size = lookup.vocabulary_size()
    # simple embedding size heuristic
    emb_dim = int(min(50, round(np.sqrt(vocab_size) + 1)))

    emb = layers.Embedding(input_dim=vocab_size, output_dim=emb_dim, name=f"{c}_emb")
    x = lookup(inputs[c])
    x = emb(x)
    x = layers.Reshape((emb_dim,), name=f"{c}_flat")(x)
    encoded_cats.append(x)

# --- Numeric: Normalization ---
# stack numeric into one tensor (batch, num_features)
num_stack = layers.Concatenate(name="num_concat")([inputs[c] for c in num_cols])
normalizer = layers.Normalization(name="num_norm")
# adapt expects a 2D array
normalizer.adapt(np.column_stack([X_train[c].astype("float32").values for c in num_cols]))
num_normed = normalizer(num_stack)

# --- Combine all ---
all_features = layers.Concatenate(name="all_features")(encoded_cats + [num_normed])


### Build a first DNN (simple but strong baseline)

**Define the DNN architecture (multi-class classification)**

We take the combined features and pass them through dense layers:

- Dense layers learn non-linear interactions between features

- Dropout reduces overfitting by randomly deactivating neurons during training

- Output layer: Dense(num_classes, softmax) produces a probability distribution over all hotel clusters

We compile with:

loss = sparse_categorical_crossentropy (correct for integer class labels)

optimizer = Adam (good default for DNN training)

In [8]:
x = layers.Dense(256, activation="relu")(all_features)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.2)(x)

# 100-way classification
outputs = layers.Dense(y.nunique(), activation="softmax", name="hotel_cluster")(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
)

model.summary()


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 site_name (InputLayer)      [(None, 1)]                  0         []                            
                                                                                                  
 posa_continent (InputLayer  [(None, 1)]                  0         []                            
 )                                                                                                
                                                                                                  
 user_location_country (Inp  [(None, 1)]                  0         []                            
 utLayer)                                                                                         
                                                                                              

### Convert pandas DataFrames to model input dict

Because we built a model with one input per column, Keras expects inputs as a dictionary:
{column_name: numpy_array}.

This helper function:

- Ensures categorical columns are the correct dtype (strings for object columns, ints for integer IDs)

- Ensures numeric columns are float32 for TensorFlow

- Produces train_in, val_in, and test_in

In [9]:
def df_to_model_input(Xdf: pd.DataFrame) -> dict:
    out = {}
    for c in cat_cols:
        if Xdf[c].dtype == "object":
            out[c] = Xdf[c].astype(str).values
        else:
            out[c] = Xdf[c].values
    for c in num_cols:
        out[c] = Xdf[c].astype("float32").values
    return out

train_in = df_to_model_input(X_train)
val_in   = df_to_model_input(X_val)
test_in  = df_to_model_input(X_test)


### Train with EarlyStopping and ModelCheckpoint

**Train the model with callbacks (early stopping + best weights saving)**

We train for up to N epochs, but use callbacks to train efficiently:

- EarlyStopping monitors val_loss and stops training if it stops improving (prevents overfitting and saves time).

- ModelCheckpoint saves the best weights to disk (based on lowest val_loss).

- This makes the run reproducible and lets us reuse the best model without retraining.

In [10]:
ckpt_path = "checkpoints/dnn_expedia_best.weights.h5"
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True, save_weights_only=True),
]

history = model.fit(
    train_in, y_train.values,
    validation_data=(val_in, y_val.values),
    epochs=30,
    batch_size=1024,
    callbacks=callbacks,
    verbose=1,
)


Epoch 1/30
59/59 [==============================] - 3s 32ms/step - loss: 4.1982 - val_loss: 3.7837
Epoch 2/30
59/59 [==============================] - 1s 20ms/step - loss: 3.7823 - val_loss: 3.6049
Epoch 3/30
59/59 [==============================] - 1s 20ms/step - loss: 3.6349 - val_loss: 3.5158
Epoch 4/30
59/59 [==============================] - 1s 21ms/step - loss: 3.5282 - val_loss: 3.4591
Epoch 5/30
59/59 [==============================] - 1s 21ms/step - loss: 3.4297 - val_loss: 3.4249
Epoch 6/30
59/59 [==============================] - 1s 20ms/step - loss: 3.3514 - val_loss: 3.4068
Epoch 7/30
59/59 [==============================] - 1s 20ms/step - loss: 3.2800 - val_loss: 3.3968
Epoch 8/30
59/59 [==============================] - 1s 18ms/step - loss: 3.2144 - val_loss: 3.4002
Epoch 9/30
59/59 [==============================] - 1s 18ms/step - loss: 3.1567 - val_loss: 3.4085
Epoch 10/30
59/59 [==============================] - 1s 18ms/step - loss: 3.1040 - val_loss: 3.4113


### Evaluate with MAP@5 and Hit@5

Our model outputs class probabilities for each sample (proba shape: n_samples × 100).
To evaluate as a recommender/ranker:

- We convert probabilities into top-K predicted hotel clusters (K=5).

- We must pass classes to map probability column index → actual hotel_cluster label.

Then we compute:

MAP@5 (primary): rewards correct predictions higher when ranked earlier

Hit@5 (secondary): checks if true class appears anywhere in the top-5

This matches the evaluation approach used for LightGBM, enabling a fair comparison.

In [12]:
proba = model.predict(test_in, batch_size=2048)

classes = np.sort(y_train.unique())  # mapping index -> hotel_cluster label

topk = topk_from_proba(proba, classes=classes, k=TOP_K)
y_true = y_test.values.tolist()

print("MAP@5:", mapk(y_true, topk, k=TOP_K))
print("Hit@5:", hit_rate_at_k(y_true, topk, k=TOP_K))


13/13 [==============================] - 0s 8ms/step
MAP@5: 0.24188266666666666
Hit@5: 0.42324
